In [1]:
import pandas as pd

In [102]:
f=open('camera_entity_resolution_gt.csv','r')
iter=0
clusters={}
rec_to_clust={}
for line in f:
    line=line.strip()
    line=line.split(',')
    lst=[]
    if line[0] in clusters.keys():
        lst=clusters[line[0]]
    lst.append(line[1])
    clusters[line[0]]=lst
    rec_to_clust[line[1]]=line[0]
    iter+=1
    
size={}
for c in clusters.keys():
    size[c] = len(clusters[c])

sorted_clusters = sorted(size.items(), key=lambda x:x[1], reverse=True)


print (sorted_clusters[:4])
chosen_clusters = (sorted_clusters[:4])

[('"ENTITY#44', 184), ('"ENTITY#23', 178), ('"ENTITY#18', 168), ('"ENTITY#36', 155)]


In [103]:
import json
lst=[]
for (c,size) in chosen_clusters:
    lst.extend(clusters[c])
    
#print (lst)

record_map={}


for recordid in lst:
    recordid=recordid.replace('"','')
    (source,r_id) = recordid.split('//')
    #print (source,r_id)
    f=open('./2013_camera_specs/'+source+'/'+r_id+'.json','r')
    json_obj = json.load(f)
    record_map[recordid] = json_obj
    #print(json_obj['<page title>'])
    
    
record_lst=list(record_map.keys())




In [184]:
import random

random.seed(1)
import re
def clean_text(t):
    #t=t.replace('Price','')
    
    #t=t.replace('India','')
    
    #t=t.replace('Bangalore','')
    #t=t.replace('Hyderabad','')
    if 'Nikon' in t:
        t = re.sub(r'\b\w{1,4}\b', '', t)
        t=t.replace('India','')
        t=t.replace('|','')
    
        t=t.replace('Bangalore','')
        t=t.replace('Hyderabad','')
        t=t.replace('Camera   Battery   Charger','')
        t=t.replace('Digital','')
        t=t.replace('Camera','')
        t=t.replace('135mm','')
        t=t.replace('Black','')
        t=re.sub(' +', ' ', t)
    if 'Canon' in t:
        t=t.replace('18 0 MP Digital SLR Camera','')
        t=t.replace('India','')
        t=t.replace('|','')
    
        t=t.replace('Bangalore','')
        t=t.replace('Hyderabad','')
        t=t.replace('Camera   Battery   Charger','')
        t=t.replace('Digital','')
        t=t.replace('Camera','')
        t=t.replace('135mm','')
        t=t.replace('Black','')
        t=re.sub(' +', ' ', t)

    
    return t
def get_distance(r1,r2):
    t1=clean_text(r1['<page title>'])
    t2=clean_text(r2['<page title>'])


    words_doc1 = set(t1.lower().split()) 
    words_doc2 = set(t2.lower().split())
    
    
    if 'nikon' in words_doc1 and 'canon' in words_doc2:
        return 1
    
    if 'nikon' in words_doc2 and 'canon' in words_doc1:
        return 1
    
    
    
    intersection = words_doc1.intersection(words_doc2)
    union = words_doc1.union(words_doc2)
        
    return 1 - float(len(intersection)) / len(union)


def assign (centers,records,record_map,assignment,distance):
    
    for r in records:
        #print (r)
        if r not in assignment.keys():
            assignment[r] = centers[0]

            distance [r] = get_distance (record_map[r],record_map[centers[0]])
            continue
            
        min_dist=distance[r]
        
        new_dist = get_distance (record_map[r],record_map[centers[-1]])
        if new_dist<min_dist:
            distance [r] = new_dist
            assignment [r] = centers[-1]
        
    return assignment,distance
    
def k_center_clustering(records,record_map,k):
    i=0
    centers=[records[random.randint(0,len(records))]]
    assignment = {}
    distance ={}
    print ("chosen this",centers[0])
    while len(centers)<k:
        
        #assign
        (assignment,distance) = assign(centers,records,record_map, assignment, distance)
        
        #Pick a center, which is the max value of distance
        
        sorted_distance = sorted(distance.items(), key=lambda x:x[1], reverse=True)
        
        newcenter = sorted_distance [0][0]
        print ("chosen this",newcenter)
        centers.append(newcenter)
        i+=1
    
    (assignment,distance) = assign(centers,records,record_map, assignment, distance)
    
    return centers,assignment

In [185]:
centers,assignment = k_center_clustering(record_lst,record_map,4)

chosen this www.ebay.com//54560
chosen this www.ebay.com//56744
chosen this www.price-hunt.com//942
chosen this www.mypriceindia.com//732


In [186]:
def get_clusters(assignment):
    clusters={}
    for r in assignment.keys():
        lst=[]
        if assignment[r] in clusters.keys():
            lst=clusters[assignment[r]]
        lst.append(r)
        clusters[assignment[r]]=lst
    return clusters

In [187]:
clusters=get_clusters(assignment)
tp=0
fp=0
for c in clusters.keys():
    curr=clusters[c]
    i=0
    while i<len(curr):
        j=i+1
        while j<len(curr):
            if rec_to_clust[curr[i]+'"'] == rec_to_clust[curr[j]+'"']:
                tp+=1
            else:
                fp+=1
            
            j+=1
        i+=1
            
print (tp,fp,tp*1.0/(tp+fp))
        

44137 40516 0.5213873105501282


In [188]:
correct=0
incorrect=0

for r in record_lst:
    center_entity_id = rec_to_clust[assignment[r]+'"']
    curr_id = rec_to_clust[r+'"']
    if curr_id==center_entity_id:
        correct+=1
    else:
        incorrect+=1
        print (clean_text(record_map[r]['<page title>']))
        print (clean_text(record_map[assignment[r]]['<page title>']))
        
    #print (curr_id,center_entity_id)
    
print(incorrect*1.0/(incorrect+correct))


for c in centers:
    print (record_map[c])
    
#Remove tokens which are 3 characters or less! this will make eos ones fail!
#What profiles to use? Fraction of 3 character words!

Nikon D3200 ( ) Price , , , Delhi, Chennai, Mumbai, , Kolkatta
Nikon D3100 (- & - ) Price , , , Delhi, Chennai, Mumbai, , Kolkatta
Nikon D3200 / -300mm - Price comparison & reviews - s - Australia
Nikon D3100 (- & - ) Price , , , Delhi, Chennai, Mumbai, , Kolkatta
Nikon D3200 price , Specs Review Valid Delhi, Mumbai, , , Chennai, Kolkata, Ahmedabad, Surat Price-
Nikon D3100 (- & - ) Price , , , Delhi, Chennai, Mumbai, , Kolkatta
Nikon D3200 / - & -300mm - Price comparison & reviews - s - Australia
Nikon D3100 (- & - ) Price , , , Delhi, Chennai, Mumbai, , Kolkatta
Nikon D3200 / -200mm - Price comparison & reviews - s - Australia
Nikon D3100 (- & - ) Price , , , Delhi, Chennai, Mumbai, , Kolkatta
Nikon D3200 / -140mm - Price comparison & reviews - s - Australia
Nikon D3100 (- & - ) Price , , , Delhi, Chennai, Mumbai, , Kolkatta
Nikon D3200 (- ) Price , , , Delhi, Chennai, Mumbai, , Kolkatta
Nikon D3100 (- & - ) Price , , , Delhi, Chennai, Mumbai, , Kolkatta
Nikon D3200 (- - & - ) Price 

In [189]:
#0.62 if we apply some cleaning steps on Nikon and others at the start
#0.54 if we apply all the cleaning at the start

In [196]:
rec_to_clust[centers[0]+'"']
(record_map[centers[0]]['<page title>'])

'Nikon D3200 DSLR Camera Body w Battery Pack and Charger | eBay'

In [191]:
rec_to_clust[centers[1]+'"']
record_map[centers[1]]['<page title>']

'Canon EOS 7D 18 0 MP Digital SLR Camera with EF s 18 135mm F 3 5 5 6 Lens 013803117493 | eBay'

In [192]:
rec_to_clust[centers[2]+'"']
record_map[centers[2]]['<page title>']

'Canon EOS 60D DSLR Camera best price in India 2014, Specs and Review | Valid in Delhi, Mumbai, Bangalore, Hyderabad, Chennai, Kolkata, Ahmedabad, Surat | Price-Hunt'

In [197]:
rec_to_clust[centers[3]+'"']
(record_map[centers[3]]['<page title>'])

'Nikon D3100 (18-55 mm & 55-200 mm) Price In India, Bangalore, Hyderabad, Delhi, Chennai, Mumbai, Pune, Kolkatta'